### RAG Pipelines - Data Ingestion to Vector DB pipeline

In [7]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\shekh\AppData\Local\Temp\ipykernel_21348\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [8]:
### Read all pdf inside the directory and its subdirectories

def process_pdf_directory(directory_path):

    """Process all PDF files in the specified directory and its subdirectories."""
    all_documents = []
    pdf_dir = Path(directory_path)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))  # Use glob to find all PDF files recursively

    print(f"Found {len(pdf_files)} PDF files in the directory and its subdirectories.")

    for pdf_file in pdf_files:
        print(f"Processing file: {pdf_file.name}")
        try:
            pdf_loader = PyPDFLoader(str(pdf_file))
            documents = pdf_loader.load()

            for document in documents:
                # Add metadata
                document.metadata["source"] = str(pdf_file.name)
                document.metadata["file_type"] = 'pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages from {pdf_file.name}\n\n")
        
        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")
    
    
    print(f"Total documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_docs = process_pdf_directory("../data/")


Found 5 PDF files in the directory and its subdirectories.
Processing file: CATApplication.pdf
Loaded 10 pages from CATApplication.pdf


Processing file: FinancialLiterarcyCourse.pdf
Loaded 1 pages from FinancialLiterarcyCourse.pdf


Processing file: ResumeJ.pdf
Loaded 2 pages from ResumeJ.pdf


Processing file: Shubham_Shekhawat_Resume_ANZ.pdf
Loaded 1 pages from Shubham_Shekhawat_Resume_ANZ.pdf


Processing file: SWE_Resume_Template.pdf
Loaded 2 pages from SWE_Resume_Template.pdf


Total documents loaded: 16


In [9]:
all_pdf_docs

[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-08-26T21:13:30+05:30', 'author': 'Sandeep Singh Shekhawat', 'moddate': '2026-08-26T21:13:30+05:30', 'title': 'CAT 2026', 'source': 'CATApplication.pdf', 'total_pages': 10, 'page': 0, 'page_label': '1', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-08-26T21:13:30+05:30', 'author': 'Sandeep Singh Shekhawat', 'moddate': '2026-08-26T21:13:30+05:30', 'title': 'CAT 2026', 'source': 'CATApplication.pdf', 'total_pages': 10, 'page': 1, 'page_label': '2', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-08-26T21:13:30+05:30', 'author': 'Sandeep Singh Shekhawat', 'moddate': '2026-08-26T21:13:30+05:30', 'title': 'CAT 2026', 'source': 'CATApplication.pdf', 'total_pages': 10, 'page': 2, 'page_label': '3', 'file_type': 

In [10]:
### Text split get into chunks

def split_documents(documents, chunk_size=100, chunk_overlap=50):
    """Split documents into chunks for better rag performance."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    if split_docs:
        print("Examole of a chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")  # Print first 200 characters of the first chunk
        print(f"Metadata: {split_docs[0].metadata}")
    return split_docs

chunked_documents = split_documents(all_pdf_docs)


Split 16 documents into 120 chunks.
Examole of a chunk:
Content: .......................................................
SANDEEP SINGH SHEKHAWAT
06-09-2026...
Metadata: {'producer': 'OpenPDF 1.3.30', 'creator': 'JasperReports Library version 6.20.5-3efcf2e67f959db3888d79f73dde2dbd7acb4f8e', 'creationdate': '2026-09-06T13:19:55+05:30', 'moddate': '2026-09-06T13:19:55+05:30', 'source': 'FinancialLiterarcyCourse.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'file_type': 'pdf'}


In [11]:
chunked_documents

[Document(metadata={'producer': 'OpenPDF 1.3.30', 'creator': 'JasperReports Library version 6.20.5-3efcf2e67f959db3888d79f73dde2dbd7acb4f8e', 'creationdate': '2026-09-06T13:19:55+05:30', 'moddate': '2026-09-06T13:19:55+05:30', 'source': 'FinancialLiterarcyCourse.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'file_type': 'pdf'}, page_content='.......................................................\nSANDEEP SINGH SHEKHAWAT\n06-09-2026'),
 Document(metadata={'producer': 'OpenPDF 1.3.30', 'creator': 'JasperReports Library version 6.20.5-3efcf2e67f959db3888d79f73dde2dbd7acb4f8e', 'creationdate': '2026-09-06T13:19:55+05:30', 'moddate': '2026-09-06T13:19:55+05:30', 'source': 'FinancialLiterarcyCourse.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'file_type': 'pdf'}, page_content='SANDEEP SINGH SHEKHAWAT\n06-09-2026\n ɱधानमंɬी िवकिसत भारत रोजगार योजना\nकमǺचारी भिवȜय िनिध संगठन'),
 Document(metadata={'producer': 'OpenPDF 1.3.30', 'creator': 'JasperReports Library version 6.20.5-

### Embedding & Vector Store DB

In [12]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\shekh\AI_Project\TradRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer and manages storage in ChromaDB."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
            Initialize the EmbeddingManager with a specified SentenceTransformer model.

            Args:
                model_name (str): The name of the SentenceTransformer model to use for embeddings.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentencceTransformer model."""
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully, Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error Loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
            Generate embeddings for a list of texts...
            
            Args:
                texts: List of text strings to embed.
                
            Returns:
                A numnpy array of embeddings with shape (len(texts), embedding_dim).
        """
        if not self.model:
            raise ValueError("Model is not loaded.")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, convert_to_numpy=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

### Initilixe embedding manager
embedding_manager = EmbeddingManager()
embedding_manager


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4790.46it/s]


Model loaded successfully, Embedding dimension: 384


### VectorStore

In [14]:
class VectorStore:
    """ Manages a vector store using ChromaDB for storing and retrieving document embeddings. """

    def __init__(self, collection_name: str = "pdf_documents", persist_directory : str = "../data/vetor_store"):
        """
           Initialize the vector store

           Args:
               collection_name: Name of the ChromaDB collection
               persist_directory: Directory to persist the vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize  ChromaDB client and collection"""

        try:
            #create persist chromadb client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                    name=self.collection_name,
                    metadata={"description":"PDF document embeddings for RAG"}
                    )
            print(f"Vector store initialized collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print("Error initializing vector store: {e}")
            raise

    def add_documents(self, documents:List[Any], embeddings:np.ndarray):
        """ 
            Add documents and their embeddings to the  vector store
            
            Args: 
                documents: List of langchain documents
                embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("No. of documents must no. of embeddings!")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadata = []
        document_txt = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            #Generate unique id
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            meta = dict(doc.metadata)
            meta['doc_index'] = i
            meta['context_length'] = len(doc.page_content)
            metadata.append(meta)

            #Document content
            document_txt.append(doc.page_content)

            # Embedding
            embeddings_list.append(embeddings.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings,
                metadatas=metadata,
                documents=document_txt,
            )
            print(f"Successfully added {len(documents)} documents to vector store.")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e} ")
            raise

vectorstore = VectorStore()
vectorstore

Vector store initialized collection: pdf_documents
Existing documents in collection: 600


In [15]:
### convert text to embeddings
texts = [doc.page_content for doc in chunked_documents]

## Generate embeddings for the chunked documents
embeddings = embedding_manager.generate_embeddings(texts)

## Store into vector db
vectorstore.add_documents(chunked_documents, embeddings)
vectorstore

Generating embeddings for 120 texts...
Generated embeddings with shape: (120, 384)
Adding 120 documents to vector store...
Successfully added 120 documents to vector store.
Total documents in collection: 720


### RAG Retriever pipeline from VectorStore

In [16]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store and provides relevant document chunks for RAG."""
    def __init__(self, vectorstore: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the RAGRetriever with a vector store and an embedding manager.

        Args:
            vectorstore: An instance of the VectorStore class for document retrieval.
            embedding_manager: An instance of the EmbeddingManager class for generating query embeddings.
        """
        self.vectorstore = vectorstore
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vectorstore.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [17]:
rag_retriever

In [22]:
rag_retriever.retrieve("finance")

Retrieving documents for query: 'finance'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_c41fd4f4_9',
  'content': 'in finance, accounting and banking concepts. Experienced',
  'metadata': {'doc_index': 9,
   'source': 'Shubham_Shekhawat_Resume_ANZ.pdf',
   'creationdate': '2026-09-10T10:08:30+05:30',
   'creator': 'Microsoft® Word 2021',
   'moddate': '2026-09-10T10:08:30+05:30',
   'producer': 'Microsoft® Word 2021',
   'page_label': '1',
   'author': 'Un-named',
   'page': 0,
   'total_pages': 1,
   'file_type': 'pdf',
   'context_length': 56},
  'similarity_score': 0.18198400735855103,
  'distance': 0.818015992641449,
  'rank': 1},
 {'id': 'doc_856f7f42_9',
  'content': 'in finance, accounting and banking concepts. Experienced',
  'metadata': {'creationdate': '2026-09-10T10:08:30+05:30',
   'creator': 'Microsoft® Word 2021',
   'author': 'Un-named',
   'file_type': 'pdf',
   'source': 'Shubham_Shekhawat_Resume_ANZ.pdf',
   'page_label': '1',
   'doc_index': 9,
   'moddate': '2026-09-10T10:08:30+05:30',
   'page': 0,
   'producer': 'Microsoft® Word 2021',
 

### Integrate VectorDB Context pipeline with LLM Output 

In [68]:
### Simple RAG pipeline with gemini api
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError("GEMINI_API_KEY not found in environment variables. Please set it in the .env file.")

llm = ChatGoogleGenerativeAI(model="gemini-flash-lite", google_api_key=api_key)

### Simole RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    ## retreive the context
    results=retriever.retrieve(query, top_k=top_k)
    context="\n\n".join([doc['context'] for doc in results]) if results else "No relevant context found."
    print(f"Context retrieved:\n{context}\n\n")

    if not context:
        return "No relevant context found."

    ## generate response using groq llm
    prompt=f"Answer the following question based on the context provided:\n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"
    
    response=llm.invoke([prompt.format(context=context, query=query)])
    return response.content

In [69]:
answer = rag_simple("strong foundation in finance,", rag_retriever, llm)

Retrieving documents for query: 'strong foundation in finance,'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Context retrieved:
No relevant context found.




GoogleModelNotFoundError: Error calling model 'gemini-flash-lite' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-flash-lite is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

In [77]:
### Simple RAG pipeline with Groq LLM
import os
from langchain_groq import ChatGroq
from dotenv import find_dotenv, load_dotenv

dotenv_path = find_dotenv(usecwd=True)
if dotenv_path:
    load_dotenv(dotenv_path, override=True)

groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("GROQ_API_KEY is missing. Add it to the project .env file, then rerun this cell.")

llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="gemma2-9b-it",
    temperature=0.1,
    max_tokens=512,
)

### Simole RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    ## retreive the context
    results=retriever.retrieve(query, top_k=top_k)
    context="\n\n".join([doc['context'] for doc in results]) if results else "No relevant context found."
    print(f"Context retrieved:\n{context}\n\n")

    if not context:
        return "No relevant context found."

    ## generate response using groq llm
    prompt=f"Answer the following question based on the context provided:\n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"
    
    response=llm.invoke([prompt.format(context=context, query=query)])
    return response.content

In [78]:
answer = rag_simple("strong foundation in finance,", rag_retriever, llm)

Retrieving documents for query: 'strong foundation in finance,'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Context retrieved:
No relevant context found.




AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}

### Enhanced RAG pipeline

In [80]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Finanace", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Finanace'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Answer: No relevant context found.
Sources: []
Confidence: 0.0
Context Preview: 


In [81]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is attention is all you need", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is attention is all you need'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}